<h1>Silver Bus Matched<h1>

<h2>Imports<h2>

In [71]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import DB_CONFIG

<h2>Harvesine Function<h2>

In [72]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))

<h2>Bus Matched Table<h2>

<h3>Query to get live feed and master schedule<h3>

In [73]:
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
query_feed = """
        SELECT gps_id, trip_id, lat as gps_lat, lon as gps_lon, time_gps, vehicle_number
        FROM silver.bus_live_feed
        WHERE etl_timestamp >= NOW() - INTERVAL '5 minutes'
    """
query_schedules = """
        SELECT *
        FROM silver.master_schedule
        WHERE trip_id = ANY(%(trip_ids)s)
    """

<h3>Extracting to df<h3>

In [74]:
df_feed = pd.read_sql(query_feed, engine)
trip_ids = df_feed['trip_id'].dropna().unique().tolist()
df_schedules = pd.read_sql(query_schedules, engine, params={"trip_ids": trip_ids})

<h2>Matching Buses with Master Schedule<h2>

In [75]:
df_merged = pd.merge(df_feed, df_schedules, on='trip_id', how='inner')
df_merged['distance'] = haversine_vectorized(
    df_merged['gps_lat'], df_merged['gps_lon'],
    df_merged['stop_lat'], df_merged['stop_lon']
)

<h3>Choosing closest scheduled point for each live feed record<h3>

In [76]:
df_merged = df_merged.sort_values(["vehicle_number","distance"])\
                .drop_duplicates(subset=["vehicle_number"], keep="first")


<h3>Writing to DB<h3>

In [ ]:
df_merged.to_sql('bus_matched', engine, schema='silver', if_exists='replace', index=False)

0